# jpabm on Colab: reproduce the IPSS validation and run a scenario

Run the cells in order. About 2–4 minutes on the free CPU runtime. The first cell installs the package from GitHub. Replace `miraemog` with the account that hosts the repository. If you only have `jpabm.zip`, upload it when asked.

In [ ]:
import importlib.util, subprocess, sys, os
REPO = 'https://github.com/miraemog/jpabm.git'   # change to your repository
if importlib.util.find_spec('jpabm') is None:
    if 'miraemog' not in REPO:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'git+' + REPO])
    else:
        if not os.path.isdir('jpabm'):
            from google.colab import files
            up = files.upload()                      # choose jpabm.zip
            import zipfile; zipfile.ZipFile(next(iter(up))).extractall('.')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', './jpabm'])
import jpabm; print('jpabm', jpabm.__version__)


## 1. Validation table (model vs IPSS 2023, fertility- and mortality-medium)

In [ ]:
from jpabm.validate import validate, summarize, pension_check
rows, R = validate(n_runs=12, n_agents=100_000)
import pandas as pd
df = pd.DataFrame(rows).set_index('year')[['pop_model','pop_ipss','dep_model','dep_ipss','old_model','old_ipss','work_model','work_ipss','kid_model','kid_ipss']].round(1)
display(df); print(summarize(rows)); print(pension_check(R))

## 2. Policy scenarios on equal footing (+100k people/year)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from jpabm import Params, run_many, data as D
N = 60_000
sc = {'baseline': Params(n_agents=N),
      'immigration +100k/yr': Params(n_agents=N, net_immigration=D.IPSS_FOREIGN_NET_INFLOW + 100_000),
      'births +100k/yr (TFR +0.18)': Params(n_agents=N, tfr_delta=0.18),
      'retirement age 68': Params(n_agents=N, retirement_age=68)}
res = {k: run_many(p, 10, 2000) for k, p in sc.items()}
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
yrs = res['baseline']['year']
for k, r in res.items():
    ax[0].plot(yrs, np.nanmean(r['pop'], 0)/1e6, label=k); ax[1].plot(yrs, np.nanmean(r['dependency'], 0)); ax[2].plot(yrs, np.nanmean(r['replacement'], 0))
ax[0].plot(D.YEARS, [D.IPSS_POP_K[y]/1e3 for y in D.YEARS], 'k--', label='IPSS')
ax[1].plot(D.YEARS, [D.IPSS_DEPENDENCY[y] for y in D.YEARS], 'k--')
for a, t in zip(ax, ['Population (M)', 'Dependency index', 'Pension pressure index (%)']): a.set_title(t)
ax[0].legend(fontsize=8); plt.tight_layout(); plt.show()

## 3. Economy and public finances (new in v0.2)
The same model now carries a fiscal block: real GDP from labour input and productivity, social security by age structure, the pension replacement rate with the macro-economic slide (floor 50%), the national primary balance, ordinary government bonds, and interest payments. Scope is the national general account only. Growth does not depend on tax, interest rates or inflation.

In [ ]:
import numpy as np, pandas as pd
from jpabm import Params, run_many, data as D
N, R = 60_000, 10
F = D.IPSS_FOREIGN_NET_INFLOW
sc = {'baseline': Params(n_agents=N),
      'immigration +100k/yr': Params(n_agents=N, net_immigration=F+100_000),
      'pension age 68': Params(n_agents=N, retirement_age=68),
      'consumption tax 15%': Params(n_agents=N, tax_consumption=15),
      'long-term yield 4%': Params(n_agents=N, long_yield=4.0),
      'inflation 3%, yield 2.5%': Params(n_agents=N, inflation=3.0),
      'inflation 3%, yield 3.5%': Params(n_agents=N, inflation=3.0, long_yield=3.5),
      'productivity 1.5%': Params(n_agents=N, productivity=1.5)}
rows = []
for k, p in sc.items():
    r = run_many(p, R, 100); m = lambda key, i: float(np.nanmean(r[key][:, i]))
    rows.append(dict(scenario=k, real_gdp_2070=m('f_real', 50), debt_pct_gdp_2070=m('f_debtgdp', 50),
                     interest_over_tax_2050=m('f_inttax', 30), primary_balance_2050=m('f_pb', 30), replacement_2060=m('replacement', 40)))
pd.DataFrame(rows).set_index('scenario').round(1)

## 4. Check the web simulator against this model
The browser version is a JavaScript port. `tests/test_model.py::test_python_matches_javascript_reference` compares its baseline with this Python model (debt ratio, real GDP, benefits, dependency ratio, population) within sampling error. The comparison table on the web page is generated from `scripts/experiments.py`, so both use the same numbers.

In [ ]:
r = run_many(Params(n_agents=100_000), 10, 41)
print({k: round(float(np.nanmean(r[k][:, i])), 1) for k, i in [('f_debtgdp', 50), ('f_real', 50), ('f_benpct', 50), ('dependency', 50)]})
print('JavaScript reference:', __import__('json').load(open('jpabm/tests/js_reference.json')))